# stochastic-rs — CUDA back-end check on a Colab GPU

Runs the native CUDA tests of `stochastic-rs-stochastic` (the fGN cuFFT + Philox pipeline and the Euler engine kernels) on the notebook's GPU, so the `cuda-native` feature can be validated without a CUDA machine of your own.

Before running: **Runtime → Change runtime type → T4 GPU** (any NVIDIA GPU works). The first cargo build takes 10–20 minutes on Colab's two cores; the tests themselves take seconds.

Cells: 1 GPU + toolkit check · 2 Rust toolchain · 3 clone · 4 native CUDA tests · 5 (optional) CubeCL CUDA runtime · 6 (optional) Python wheel with `device="cuda-native"`.

In [ ]:
# 1. The GPU and the CUDA toolkit. The driver's CUDA version (nvidia-smi, top right)
#    should be >= the toolkit's (nvcc): NVRTC emits PTX for the toolkit's version and
#    an older driver cannot JIT it (CUDA_ERROR_UNSUPPORTED_PTX_VERSION). Colab ships
#    them matched; if they differ, see the note at the end.
!nvidia-smi
!nvcc --version | tail -2

In [ ]:
# 2. Rust (stable, minimal profile). PATH is extended for every later cell.
import os
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
os.environ["CARGO_TERM_COLOR"] = "never"
!cargo --version && rustc --version

In [ ]:
# 3. The repository. REF is a branch, tag or commit; main is the default.
REF = "main"
!rm -rf stochastic-rs && git clone --quiet --depth 1 --branch {REF} https://github.com/rust-dd/stochastic-rs.git
%cd stochastic-rs
!git log --oneline -1

In [ ]:
# 4. Native CUDA tests: fGN (cuFFT + Philox; shapes, moments against the CPU path,
#    seed reproducibility regardless of launch history) and the Euler engine (GBM
#    moments, CIR positivity, f32/f64 agreement, chunked batches). ~10-20 min to build.
!cargo test -p stochastic-rs-stochastic --features cuda-native --lib -- cuda_native 2>&1 | grep -E "^test |^test result|error|panicked" 

In [ ]:
# 5. (Optional) the CubeCL CUDA runtime as well: the portable kernels on the same GPU,
#    plus the Metal ≡ CubeCL-style agreement checks that exist for CUDA. Adds a few
#    minutes of build time. Set RUN_CUBECL = True to run.
RUN_CUBECL = False
if RUN_CUBECL:
    !cargo test -p stochastic-rs-stochastic --features cuda-native,cubecl-cuda --lib -- cuda_native cubecl 2>&1 | grep -E "^test |^test result|error|panicked" 

In [ ]:
# 6. (Optional) the Python module with the CUDA back-end: builds the wheel (release,
#    20-30 min on Colab) and runs the device-related pytest cases with
#    device="cuda-native". Set RUN_PYTHON = True to run.
RUN_PYTHON = False
if RUN_PYTHON:
    !pip -q install maturin pytest numpy
    !maturin develop --release --features cuda-native 2>&1 | tail -2
    !python -m pytest -q stochastic-rs-py/tests/test_stochastic.py -k "device or euler_paths or probe" 2>&1 | tail -8
    import importlib
    import numpy as np
    sr = importlib.import_module("stochastic_rs")
    print(sr.probe_device("cuda-native"))
    paths = sr.euler_paths("gbm", [0.05, 0.2], 100.0, 253, 1.0, 20_000, seed=7, device="cuda-native")
    print(paths.shape, paths.dtype, "terminal mean / forward =", paths[:, -1].mean() / (100.0 * np.exp(0.05)))


## Reading the result

Every `cuda_native*` test prints `ok` and the summary line ends in `0 failed`. Two things are worth a look if something is red:

- `CUDA_ERROR_UNSUPPORTED_PTX_VERSION` on module load: the toolkit (nvcc) is newer than the driver. Either pick a runtime whose versions match, or install the toolkit matching `nvidia-smi`'s CUDA version (`apt-get install cuda-toolkit-12-x`) and rerun cell 4.
- `device unavailable: CudaContext: ...`: no GPU in this runtime. Change the runtime type to a GPU and run from cell 1.

The same commands run on any CUDA machine; `STOCHASTIC_RS_DEVICE=n` picks the GPU when there are several.